# Scenario comparison at `n*`

Each scenario is simulated for its own replication count from decision 9 in `docs/assumptions.md`. The counts come from `data/queue_delay_confidence_intervals.csv`; replication seeds are `1..n*` in every scenario. The table reports the mean of every field in `PerformanceMetrics` across those replications, plus 95% t interval endpoints for the two average queue delays.

Delays and time in system are in minutes. Queue lengths and patient counts are average patients per replication. Utilisation and waiting shares are proportions from 0 to 1. The `nurse_rho` and `rooms_rho` columns are offered load, calculated from the expected service times below; simulated utilisation is also shown separately.

## Utilisation and stability

The expected triage service time is $(2+4+6)/3 = 4$ minutes. The expected treatment time is $0.5(15)+0.3(25)+0.2(45)=24$ minutes. With one nurse, $\rho_{nurse}=4/IA$; with $R$ rooms, $\rho_{rooms}=24/(IA\times R)$.

A queue is stable only when $\rho<1$. At $\rho=1$ the average workload equals capacity, leaving no spare capacity; this critical case is not stable. The tandem ER is stable only if both stages are stable. The `system_stability` column applies this strict rule.

- **Base:** nurse $\rho=1.33$ and rooms $\rho=1.33$. Both receive work faster than they can serve it, so both queues are unstable.
- **A:** nurse $\rho=0.67$ is stable, while rooms $\rho=1.00$ are critical. The room stage has no capacity margin and is the bottleneck; its mean delay is 41.74 min versus 3.83 min at triage.
- **B:** nurse $\rho=0.67$ and rooms $\rho=0.50$. Both stages have spare capacity; this is the only stable scenario and it has the smallest room delay and overall time in system.
- **C:** nurse $\rho=1.33$ and rooms $\rho=2.00$. Both stages are overloaded, with room demand twice its service capacity. It has the largest room delay (231.93 min) and time in system (369.55 min).
- **D:** nurse $\rho=1.33$ is overloaded and rooms $\rho=1.00$ are critical. The small observed room delay does not make the system stable: the overloaded nurse limits flow to rooms, and this one-day run starts empty.

## Interpretation

Scenario B performs best because both resources have capacity slack. Scenario A shows a room bottleneck despite a stable nurse. Increasing room count in D keeps room waits low but does not remove the overloaded triage nurse, so triage delays remain high. C is worst overall because both stages are overloaded, especially the room stage. Since each run is a one-day terminating simulation that starts empty, measured delays in unstable scenarios are biased downward when the longest-waiting patients have not started service by minute 1440. The precision-based `n*` controls the estimated mean's interval width; it does not make an unstable scenario stable.


In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
from scenario_comparison import compare_scenarios


In [2]:
comparison = compare_scenarios()
comparison.to_csv(
    ROOT / "data" / "scenario_comparison.csv",
    index=False,
    float_format="%.10g",
)
pd.set_option("display.max_columns", None)
comparison.round(3)


scenario,n_rooms,mean_interarrival,run_length,n_star,nurse_rho,rooms_rho,system_stability,average_triage_delay,average_room_delay,average_time_in_system,average_triage_queue_length,average_room_queue_length,average_number_in_system,nurse_utilisation,rooms_utilisation,share_waiting_more_than_30_minutes,share_triage_wait_more_than_30_minutes,share_room_wait_more_than_30_minutes,n_arrived,n_triage_started,n_room_started,n_departed,triage_delay_ci_low,triage_delay_ci_high,room_delay_ci_low,room_delay_ci_high
Base,6,3,1440,162,1.333,1.333,not stable,181.364,19.302,221.108,60.553,4.797,72.119,0.995,0.962,0.925,0.908,0.235,479.272,358.315,349.389,343.457,176.283,186.444,17.230,21.375
A,4,6,1440,114,0.667,1.000,not stable,3.830,41.738,73.169,0.639,7.085,12.123,0.658,0.935,0.531,0.003,0.490,237.570,237.061,226.228,222.377,3.624,4.035,36.438,47.038
B,8,6,1440,534,0.667,0.500,stable,4.040,0.036,31.972,0.682,0.006,5.282,0.663,0.491,0.004,0.004,0.000,239.566,238.942,238.303,234.451,3.928,4.151,0.031,0.041
C,4,3,1440,17,1.333,2.000,not stable,172.546,231.929,369.549,57.072,58.176,120.193,0.994,0.988,0.943,0.858,0.915,473.647,358.059,237.941,233.941,156.035,189.057,220.306,243.551
D,8,3,1440,121,1.333,1.000,not stable,180.742,0.254,205.408,60.570,0.063,67.547,0.995,0.740,0.912,0.911,0.000,479.711,358.380,357.281,351.215,174.777,186.706,0.222,0.285
